# Mercado Laboral y Salarios en Tecnología (Chile y Latinoamérica 2024–2026)
**Certificado Profesional de Análisis de Datos de Google • Caso Práctico 3 (Pista B)**  
**Autor:** Manuel Yáñez Orellana

Este cuaderno reproduce el flujo analítico completo (**Fases 3, 4 y 5**) sobre datos limpios manteniendo las **4 poblaciones separadas** y aplicando el umbral **`N >= 30`** (**Curso 4, Módulo 1: Teorema del Límite Central al 95% de confianza**).


In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams["text.parse_math"] = False
sns.set_theme(style="whitegrid")

# Detecta automáticamente ruta local en el repositorio o entorno Kaggle Input
CANDIDATE_DIRS = [
    Path("03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("../03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("/Users/leunam/Library/CloudStorage/GoogleDrive-manuelya@gmail.com/Mi unidad/Portafolio Google Data Analytics/03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("/kaggle/input/mercado-tech-chile-latam-limpios"),
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if d.exists()), CANDIDATE_DIRS[0])
print("Directorio activo de datos limpios:", DATA_DIR.resolve())


In [ ]:
gob_vac = pd.read_csv(DATA_DIR / "GetOnBoard_Vacantes_2026_20261111_v01_clean.csv")
gob_hab = pd.read_csv(DATA_DIR / "GetOnBoard_Habilidades_2026_20261111_v01_clean.csv")
so_lat  = pd.read_csv(DATA_DIR / "StackOverflow_LatAm_2024_20261111_v01_clean.csv")
so_tec  = pd.read_csv(DATA_DIR / "StackOverflow_Tecnologias_2024_20261111_v01_clean.csv")
dss     = pd.read_csv(DATA_DIR / "DataScienceSalaries_2020_2024_20261111_v01_clean.csv")
ine     = pd.read_csv(DATA_DIR / "INE_ESI_2025_ReferenciaNacional_20261111_v01_clean.csv")

print(f"GOB ofertas={len(gob_vac)} (salarios válidos={int(gob_vac['salario_valido_flag'].sum())})")
print(f"SO LatAm={len(so_lat)} (salarios válidos={int(so_lat['salario_valido_flag'].sum())}, Chile={int((so_lat['pais']=='Chile').sum())})")
print(f"DSS 2020-2024={len(dss)} | INE referencia nacional CLP mediana={int(ine.loc[ine['indicador'].str.contains('Mediano', case=False), 'monto_clp_mensual'].iloc[0]):,}")


## 1. Análisis Relacional en SQLite (Curso 5)
Verificación de denominadores con `CREATE TEMP TABLE`, `LEFT JOIN`, `COUNT(DISTINCT)` y `HAVING` sin duplicación en uniones `1:N`.


In [ ]:
conn = sqlite3.connect(":memory:")
gob_vac.to_sql("vacantes", conn, index=False)
gob_hab.to_sql("vacante_habilidades", conn, index=False)
so_lat.to_sql("so_encuestados", conn, index=False)
so_tec.to_sql("so_tecnologias", conn, index=False)

q_top5_gob = """
SELECT
    h.tipo_tecnologia,
    h.tecnologia,
    COUNT(DISTINCT h.job_id) AS n_ofertas,
    ROUND(100.0 * COUNT(DISTINCT h.job_id) / (SELECT COUNT(DISTINCT job_id) FROM vacantes), 2) AS pct_demanda
FROM vacante_habilidades AS h
GROUP BY h.tipo_tecnologia, h.tecnologia
HAVING 100.0 * COUNT(DISTINCT h.job_id) / (SELECT COUNT(DISTINCT job_id) FROM vacantes) >= 20.0
ORDER BY pct_demanda DESC;
"""
df_top5 = pd.read_sql_query(q_top5_gob, conn)
df_top5


## 2. Estadística Descriptiva en Python (`groupby()`, `describe()`, `quantile()`, regla `N >= 30`)
Cálculo de medianas salariales USD, cuartiles `p25–p75` y validación de composición Modalidad × Seniority (`Correlación != Causalidad`).


In [ ]:
gob_sal = gob_vac[gob_vac["salario_valido_flag"] == True].copy()
bandas_exp = (
    gob_sal.groupby("seniority_nivel")["salario_mensual_usd"]
    .agg(
        n="count",
        mediana_usd="median",
        p25_usd=lambda s: s.quantile(0.25),
        p75_usd=lambda s: s.quantile(0.75),
    )
    .reset_index()
)
bandas_exp["cumple_n30"] = bandas_exp["n"] >= 30
print("=== Bandas GOB por Seniority ===")
display(bandas_exp.sort_values("mediana_usd", ascending=False))

comp_q12 = (
    gob_sal.groupby(["modalidad_grupo", "seniority_nivel"])
    .size()
    .unstack(fill_value=0)
)
print("=== Composición Modalidad x Seniority (Q12: Correlación != Causalidad) ===")
display(comp_q12)


## 3. Visualización Rápida (Paleta Apta para Daltonismo, Eje en `0%` • Curso 6)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2), dpi=140)
df_plot = df_top5.sort_values("pct_demanda", ascending=True)
colors = ["#D55E00" if t == "Microsoft Azure" else "#1B365D" for t in df_plot["tecnologia"]]
ax.barh(df_plot["tecnologia"], df_plot["pct_demanda"], color=colors, height=0.55)
ax.axvline(20.0, color="#D55E00", linestyle="--", linewidth=1.2)
for idx, r in enumerate(df_plot.itertuples()):
    ax.text(r.pct_demanda + 0.6, idx, f"{r.pct_demanda:.2f}% (N={r.n_ofertas})", va="center", fontsize=9, fontweight="bold")
ax.set_xlim(0, 42)
ax.set_xlabel("% de Ofertas Vigentes en Get on Board (N=679)")
ax.set_title("Solo 5 tecnologías superan la meta >= 20% en demanda (Chile/Remoto 2026)", loc="left", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
